# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadaFouad461/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I use five features available from the February decision window: impressions, clicks, sessions, average position, and search volume. The features are aggregated to one row per client and content item.

In [3]:
%pip -q install duckdb huggingface_hub pandas numpy

import getpass
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import login, hf_hub_download, list_repo_files

# Authentication
HF_TOKEN = getpass.getpass("Enter Hugging Face Token: ")
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

#  Download Data
all_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
feb_files = [f for f in all_files if "month=2026-02" in f and f.endswith(".parquet")]
mar_files = [f for f in all_files if "month=2026-03" in f and f.endswith(".parquet")]

local_feb_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in feb_files]
local_mar_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in mar_files]

#  Connection Setup
con = duckdb.connect()
FACT_FEB = f"read_parquet({local_feb_paths})"
FACT_MAR = f"read_parquet({local_mar_paths})"

print(" Connection & Data Ready!")

Enter Hugging Face Token: ··········


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

 Connection & Data Ready!


In [6]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date) AS active_days_feb,
    SUM(gsc_clicks) AS total_clicks_feb,
    SUM(gsc_impressions) AS total_impressions_feb,
    AVG(gsc_avg_position) AS avg_position_feb,
    CASE
        WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr_feb
FROM {FACT_FEB}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,active_days_feb,total_clicks_feb,total_impressions_feb,avg_position_feb,ctr_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,28,0.0,299.0,12.946228,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,28,6.0,733.0,6.495085,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,28,0.0,514.0,10.490023,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,28,3.0,2931.0,38.436254,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,28,2.0,970.0,9.710810,0.002062


### Feature notes

- impressions_90d — measures observed search visibility before the decision moment; missing values are handled during feature preparation.
- clicks_90d — measures observed search clicks before the decision moment; missing values are handled during feature preparation.
- sessions_90d — measures observed sessions before the decision moment; missing values are handled during feature preparation.
- avg_position — measures observed average search position before the decision moment; missing values are handled during feature preparation.
- search_volume — describes observed search demand and is available before the decision moment; missing values are handled during feature preparation.

All five features come from February and therefore precede the March outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

I deliberately add the March outcome trend_pct as a feature. Because this value is derived from the future outcome window, it should not be available at the February decision moment. A high relationship with the outcome demonstrates leakage. I then remove it and keep the honest feature set.

In [8]:
# March outcome (Target Label for March 2026)
march_label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_total_clicks
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

# Deliberate leakage test (Merging March target into February features)
leaky_frame = feature_frame.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Creating a leaky feature from future March outcome
leaky_frame["leaky_feature"] = leaky_frame["march_total_clicks"]

print("Rows with deliberate leakage test:", len(leaky_frame))
print(
    "Correlation between leaky feature and March outcome:",
    leaky_frame["leaky_feature"].corr(leaky_frame["march_total_clicks"])
)

Rows with deliberate leakage test: 134238
Correlation between leaky feature and March outcome: 1.0


## 4. What I excluded and why

- client_hash_id: excluded because it is a pseudonymous identifier, not a meaningful predictive feature.
- content_hash_id: excluded because it identifies the content item but does not describe its measurable characteristics.
- March trend_pct: excluded because it is the future outcome and using it as a feature would leak the label.
- Other March performance fields: excluded because they are not available at the February decision moment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.